# 🔍 La convolution à la main (from scratch)

**Objectif** : comprendre ce qu'un CNN fait *réellement* sur une image, sans PyTorch — juste NumPy.

> **Analogie** (chapitre 0 §10) — un filtre de convolution est une **lampe torche** qui glisse sur
> l'image et **s'allume quand elle reconnaît un motif** (un bord, un coin). Ici on va le voir.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Une image synthétique (un carré clair sur fond sombre)

In [ ]:
img = np.zeros((16, 16))
img[4:12, 4:12] = 1.0   # un carré blanc au centre
plt.imshow(img, cmap='gray'); plt.title('Image d\'entrée'); plt.axis('off'); plt.show()

## 2. Des filtres détecteurs de bords
Chaque filtre 3×3 cherche un motif précis. Le filtre vertical réagit aux **bords verticaux**.

In [ ]:
bord_vertical   = np.array([[-1,0,1],[-1,0,1],[-1,0,1]])
bord_horizontal = np.array([[-1,-1,-1],[0,0,0],[1,1,1]])
print('Filtre bord vertical:\n', bord_vertical)

## 3. La convolution, pas à pas
On fait **glisser** le filtre sur l'image ; à chaque position, on multiplie terme à terme et on somme.

In [ ]:
def convolution(image, noyau):
    h, w = image.shape
    kh, kw = noyau.shape
    sortie = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(sortie.shape[0]):
        for j in range(sortie.shape[1]):
            zone = image[i:i+kh, j:j+kw]   # la fenêtre sous la 'lampe torche'
            sortie[i, j] = np.sum(zone * noyau)   # produit terme à terme puis somme
    return sortie

carte_v = convolution(img, bord_vertical)
carte_h = convolution(img, bord_horizontal)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12,4))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('Entrée')
ax[1].imshow(carte_v, cmap='gray'); ax[1].set_title('Bords verticaux détectés')
ax[2].imshow(carte_h, cmap='gray'); ax[2].set_title('Bords horizontaux détectés')
for a in ax: a.axis('off')
plt.show()
# Observe : le filtre vertical s'allume sur les côtés gauche/droit du carré, l'horizontal sur le haut/bas.

## 4. ReLU puis pooling
**ReLU** garde ce qui est positif (l'interrupteur du chapitre 0 §3). **Max-pooling** résume chaque
zone 2×2 par son maximum (§10) : on garde *« y avait-il un bord ici ? »* en divisant la taille par 2.

In [ ]:
def relu(x): return np.maximum(0, x)

def max_pool(x, taille=2):
    h, w = x.shape
    out = np.zeros((h//taille, w//taille))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i,j] = x[i*taille:(i+1)*taille, j*taille:(j+1)*taille].max()
    return out

resume = max_pool(relu(carte_v))
print('Taille avant pooling:', carte_v.shape, '-> après:', resume.shape)
plt.imshow(resume, cmap='gray'); plt.title('Après ReLU + max-pooling'); plt.axis('off'); plt.show()

## 🎯 À toi de jouer
Crée un filtre de **flou** (moyenne 3×3, chaque coefficient = 1/9) et applique-le à l'image. Que se passe-t-il ?

In [ ]:
# Écris ta réponse ici :


<details><summary>💡 Corrigé</summary>

```python
flou = np.ones((3,3)) / 9.0
img_floue = convolution(img, flou)
plt.imshow(img_floue, cmap='gray'); plt.title('Flou'); plt.axis('off'); plt.show()
# Les bords nets deviennent des dégradés : le filtre 'moyenne' lisse l'image.
```
</details>

## ✅ À retenir
- Une **convolution** = un petit filtre qui **glisse** et **détecte un motif** (produit terme à terme + somme).
- Différents filtres détectent différents motifs (bords, textures) ; un CNN **apprend** ces filtres tout seul.
- **ReLU** garde le positif, **pooling** résume et réduit la taille → robustesse + efficacité.
- Empilés, ces blocs vont du bord → à l'œil → au visage (l'usine à étages, chapitre 0 §4).